# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring**

I'm picking this lane because I've already been working directly in this space through the Week 1-2 notebooks: building a hand-written "stale x visible" rule, comparing it against a readable decision tree using Precision@K, and seeing the starter pipeline's baseline-vs-model comparison. The problem — ranking pages by which one a reviewer should look at first — has a concrete, honest evaluation method (Precision@K) and a clear decision-support framing, which fits how the lane guide describes a "good" capstone question. It also builds directly on data I already understand (the starter CSV, `trend_direction`, `impressions_90d`, `days_since_last_update`), so I can go deeper into the warehouse release later instead of starting from scratch.

In [1]:
# (no code needed for this section - see markdown above)

## 2. The question: decision, action, cost of a wrong call

**Decision improved:** which content pages a review team should prioritize first when they only have capacity to review a limited number of pages (e.g. top 20 or top 50) out of thousands.

**Who acts on it:** a content/SEO reviewer (at FlyRank or a client team) who has limited weekly capacity and currently either reviews pages in an arbitrary order or uses a simple hand-written rule (like "stale AND visible").

**The action:** the reviewer opens the ranked queue, checks the reason codes attached to each page (e.g. `stale_visible_page`, `declining_with_demand`), and decides whether to refresh, expand, protect, prune, or monitor that page.

**Cost of a wrong recommendation:**
- **False positive** (page flagged as high-priority but wasn't actually a problem): wastes reviewer time on a page that didn't need attention - the cost is hours, not damage to the site.
- **False negative** (a genuinely declining/high-opportunity page never surfaces in the top-K): the page keeps losing visibility unnoticed, which is a larger real cost since search visibility is hard to win back.
- Because false negatives are costlier here, precision alone isn't the whole story - recall and reviewing the "just outside top-K" pages matters too, and I'll need to say so honestly rather than only optimizing precision@K.

**Why data or ML can help at all:** with thousands of pages and only a handful of observable signals per page (impressions, position, CTR, staleness, word count, trend), a human can't manually cross-reference all of them at scale - but a simple model can rank pages using patterns learned from many pages at once, and the starter pipeline already shows a learned model can outperform a fixed hand rule at Precision@50 (0.740 vs 0.240 in the starter results). That gap is the evidence this lane is worth pursuing further.

In [2]:
# (no code needed for this section - see markdown above)

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: scale and base rate
print(f"Rows: {df.shape[0]}")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")

# Number 2: how many pages are "stale and visible" - my hand-rule candidate pool
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale AND visible pages: {stale_visible} ({stale_visible/len(df):.1%} of all pages)")

# Number 3: starter pipeline result - baseline rule vs learned model at Precision@50
print("\nFrom outputs/model_report.md (starter pipeline, client-holdout validated):")
print("baseline rules      Precision@50: 0.240")
print("random forest       Precision@50: 0.740")
print("-> the learned model found ~3x more true positives in its top 50 than the fixed rule")


Rows: 30000
Declining rate: 0.542
Stale AND visible pages: 17 (0.1% of all pages)

From outputs/model_report.md (starter pipeline, client-holdout validated):
baseline rules      Precision@50: 0.240
random forest       Precision@50: 0.740
-> the learned model found ~3x more true positives in its top 50 than the fixed rule


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- Observed correlations and patterns between page signals (staleness, impressions, position, CTR) and a measurable label (declining trend, or a future-window outcome I define later).
- Directional evidence - e.g. "in this sample, pages meeting condition X were more/less likely to show label Y."
- Decision-support output - a ranked queue with reason codes that helps a reviewer prioritize limited time, not a guarantee about any individual page.
- Comparisons between a simple baseline rule and a learned model, evaluated with Precision@K on held-out data.

**What this work will never claim:**
- That it has reverse-engineered or proven anything about Google's actual ranking algorithm.
- That a refresh or edit *caused* a page to recover - that requires a real experiment (e.g. before/after with a control group), which this observational data cannot provide on its own.
- That the current proxy label (`trend_direction == "down"`, a bucket computed from the current window) is the same as a true future outcome - it's a starting point, and a stronger version of this lane would define the label as a future window (e.g. decline over the next 30 days, using only features known before that window).
- That a high-priority score means a page is broken - it means the page is a good candidate for a human to look at first, given limited review capacity.

In [4]:
# (no code needed for this section - see markdown above)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.